In [ ]:
# -*- coding: utf-8 -*-
# ---
# jupyter:
#   jupytext:
#     text_representation:
#       extension: .py
#       format_name: light
#       format_version: '1.5'
#       jupytext_version: 1.14.5
#   kernelspec:
#     display_name: Python 3 (ipykernel)
#     language: python
#     name: python3
# ---

# # Análise dos Resultados da Otimização de L0 via Algoritmos Genéticos

# ## 1. Configurações e Imports

# +
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import os
import glob
from collections import Counter

# Adicionar o diretório pai ao sys.path para importar o módulo da biblioteca
import sys
# Presumindo que o notebook está em 'examples', e a biblioteca em 'activetextclassification' no nível acima
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

try:
    from activetextclassification.visualization.ag_plots_evolution import plot_population_evolution_combined
    print("Função de plotagem 'plot_population_evolution_combined' importada.")
except ImportError as e:
    print(f"Erro ao importar 'plot_population_evolution_combined': {e}")
    print("Certifique-se que 'activetextclassification/visualization/ag_plots_evolution.py' existe e o PYTHONPATH está correto.")
    # Definir uma função placeholder para evitar que o resto do notebook quebre
    def plot_population_evolution_combined(*args, **kwargs):
        print("ERRO: plot_population_evolution_combined não pôde ser importada. Gráfico não gerado.")


# Configurações de Estilo para os Gráficos
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 7) 
plt.rcParams['axes.titlesize'] = 14 # Reduzido para subplots
plt.rcParams['axes.labelsize'] = 12 # Reduzido
plt.rcParams['xtick.labelsize'] = 10 # Reduzido
plt.rcParams['ytick.labelsize'] = 10 # Reduzido
plt.rcParams['legend.fontsize'] = 8  # Reduzido
plt.rcParams['figure.titlesize'] = 16 # Para suptitle

# ## 2. Definição de Parâmetros e Caminhos 
# (sem grandes alterações, apenas garantindo que CONVERGENCE_L0_SIZES_EXAMPLE seja usado corretamente)

# +
base_ag_results_path = "." 
l0_size_folders = glob.glob(os.path.join(base_ag_results_path, "ag_optimization_results_L0_*"))
AVAILABLE_L0_SIZES = sorted([int(os.path.basename(f).split('_')[-1]) for f in l0_size_folders if os.path.basename(f).split('_')[-1].isdigit()])
print(f"Tamanhos de L0 com pastas de resultados AG encontradas: {AVAILABLE_L0_SIZES}")

L0_SIZES_PRIMARY = [10, 50, 100, 500, 1000, 2500, 5000, 10000, 20000, 30000, 100000]
L0_SIZES_FOR_PLOTS = [s for s in L0_SIZES_PRIMARY if s in AVAILABLE_L0_SIZES]
if not L0_SIZES_FOR_PLOTS and AVAILABLE_L0_SIZES: 
    L0_SIZES_FOR_PLOTS = AVAILABLE_L0_SIZES
elif not L0_SIZES_FOR_PLOTS and not AVAILABLE_L0_SIZES:
    print("ALERTA: Nenhum L0_SIZE disponível para os gráficos de 'Curva Ótima' e 'Características'.")
    L0_SIZES_FOR_PLOTS = [10] 

# Para os gráficos de evolução da população, usaremos os exemplos da orientação
# ou o que estiver disponível.
CONVERGENCE_L0_SIZES_EXAMPLE = [s for s in [10, 50, 100, 500, 1000, 2500, 5000, 10000, 20000, 30000, 100000] if s in AVAILABLE_L0_SIZES]
if not CONVERGENCE_L0_SIZES_EXAMPLE and AVAILABLE_L0_SIZES: # Se os da orientação não existem, pega os primeiros disponíveis
    CONVERGENCE_L0_SIZES_EXAMPLE = AVAILABLE_L0_SIZES[:min(4, len(AVAILABLE_L0_SIZES))]
elif not CONVERGENCE_L0_SIZES_EXAMPLE and not AVAILABLE_L0_SIZES: # Se nada disponível
     print("ALERTA: Nenhum L0_SIZE disponível para os gráficos de 'Evolução da População'.")
     CONVERGENCE_L0_SIZES_EXAMPLE = []


print(f"Tamanhos de L0 para gráficos de 'Curva Ótima' e 'Características': {L0_SIZES_FOR_PLOTS}")
print(f"Tamanhos de L0 para gráficos de 'Evolução da População': {CONVERGENCE_L0_SIZES_EXAMPLE}")

RANDOM_SAMPLING_RESULTS_FILE = os.path.join("data", "sensibilidade", "l0_random_impact_metrics_PVBin.xlsx")
FULL_DATASET_FILE = os.path.join("..", "data", "dataset.csv") 
TEXT_COLUMN = 'nm_item'; LABEL_COLUMN = 'nm_product'
AG_BEST_L0_BASE_NAME = "ag_best_l0"; AG_DETAILED_FITNESS_BASE_NAME = "ag_detailed_fitness" 
METRIC_MAP = {"ACCURACY": "Acurácia", "F1": "Macro F1-Score"}
GOAL_MAP = {"MAXIMIZE": "Maximização", "MINIMIZE": "Minimização"}

MAX_GENERATIONS_TO_PLOT = 100 # Limite de gerações para os plots
# -
# -

# ## 3. Carregamento de Dados 
# (Mantido como na versão anterior)

# ### 3.1 Dados da Amostragem Aleatória (Sensibilidade L0)

# +
df_random_stats = None
try:
    df_random_stats = pd.read_excel(RANDOM_SAMPLING_RESULTS_FILE)
    rename_map_perf = {'L0 Size': 'L0_Size', 'Mean Accuracy': 'mean_accuracy', 'Min Accuracy': 'min_accuracy', 'Max Accuracy': 'max_accuracy', 'Mean Macro F1-score': 'mean_f1_score', 'Min Macro F1-score': 'min_f1_score', 'Max Macro F1-score': 'max_f1_score'}
    rename_map_chars = {'Avg. Total Tokens': 'avg_total_tokens', 'Avg. Distinct Tokens': 'avg_distinct_tokens', 'Avg. Unique Classes': 'avg_num_classes'}
    df_random_stats.rename(columns={**rename_map_perf, **rename_map_chars}, inplace=True)
    if 'L0_Size' in df_random_stats.columns:
        df_random_stats['L0_Size'] = pd.to_numeric(df_random_stats['L0_Size'], errors='coerce')
        df_random_stats.dropna(subset=['L0_Size'], inplace=True)
        df_random_stats.set_index('L0_Size', inplace=True)
except FileNotFoundError: print(f"ERRO: Arquivo de amostragem aleatória não encontrado: {RANDOM_SAMPLING_RESULTS_FILE}")
except Exception as e: print(f"ERRO ao carregar dados da amostragem aleatória: {e}")
# -

# ### 3.2 Dataset Completo (para análise de características dos L0s)

# +
df_full = None
try:
    df_full = pd.read_csv(FULL_DATASET_FILE)
    df_full.dropna(subset=[TEXT_COLUMN, LABEL_COLUMN], inplace=True)
except FileNotFoundError: print(f"ERRO: Dataset completo não encontrado: {FULL_DATASET_FILE}"); df_full = None 
except Exception as e: print(f"ERRO ao carregar o dataset completo: {e}"); df_full = None
# -

# ## 4. Funções Auxiliares
# (Função `load_and_aggregate_detailed_log` é crucial aqui)

# +
def get_ag_best_l0_filepath(l0_size, metric_short, goal_short):
    dir_path = os.path.join(base_ag_results_path, f"ag_optimization_results_L0_{l0_size}")
    filename = f"{AG_BEST_L0_BASE_NAME}_{metric_short.upper()}_{goal_short.upper()}.csv"
    return os.path.join(dir_path, filename)

def get_ag_detailed_fitness_filepath(l0_size, metric_short, goal_short):
    dir_path = os.path.join(base_ag_results_path, f"ag_optimization_results_L0_{l0_size}")
    filename = f"{AG_DETAILED_FITNESS_BASE_NAME}{metric_short.upper()}_{goal_short.upper()}.csv"
    return os.path.join(dir_path, filename)

def load_and_aggregate_detailed_log(l0_size, metric_short_name, goal_short_name, max_gens=None): # Adicionado max_gens
    filepath = get_ag_detailed_fitness_filepath(l0_size, metric_short_name, goal_short_name)
    metric_col_in_file = 'accuracy_on_full' if metric_short_name.upper() == 'ACCURACY' else 'f1_macro_on_full'
    try:
        df_detailed = pd.read_csv(filepath)
        if max_gens and 'generation' in df_detailed.columns: # TRUNCAR GERAÇÕES
            df_detailed = df_detailed[df_detailed['generation'] <= max_gens]
        if df_detailed.empty: return None
        if metric_col_in_file not in df_detailed.columns: return None
        df_detailed[metric_col_in_file] = pd.to_numeric(df_detailed[metric_col_in_file], errors='coerce')
        df_detailed.dropna(subset=[metric_col_in_file], inplace=True) 
        if df_detailed.empty: return None
        char_cols = ['num_tokens', 'num_distinct_tokens', 'num_classes_in_l0']
        grouped = df_detailed.groupby('generation')
        df_agg_perf = grouped[metric_col_in_file].agg(['max', 'mean', 'min']).reset_index()
        df_agg_perf.rename(columns={'max': 'max_metric', 'mean': 'avg_metric', 'min': 'min_metric'}, inplace=True)
        
        # Para características, precisamos dos dados originais antes do groupby
        # e pegar do indivíduo com melhor/pior métrica
        df_detailed_for_chars = df_detailed.copy() 

        if goal_short_name.upper() == 'MAXIMIZE':
            idx_best_worst = df_detailed_for_chars.groupby('generation')[metric_col_in_file].idxmax()
        else: # MINIMIZE
            idx_best_worst = df_detailed_for_chars.groupby('generation')[metric_col_in_file].idxmin()
        
        df_best_worst_chars_raw = df_detailed_for_chars.loc[idx_best_worst]
        
        # Selecionar apenas as colunas de características e geração
        cols_to_merge = ['generation'] + [col for col in char_cols if col in df_best_worst_chars_raw.columns]
        df_best_worst_chars = df_best_worst_chars_raw[cols_to_merge].reset_index(drop=True)

        if df_best_worst_chars.empty or not any(col in char_cols for col in df_best_worst_chars.columns):
            df_agg_final = df_agg_perf.copy()
            for col in char_cols: df_agg_final[col] = np.nan
        else:
            df_agg_final = pd.merge(df_agg_perf, df_best_worst_chars, on='generation', how='left')
        return df_agg_final
    except FileNotFoundError: return None
    except Exception: return None


def get_l0_characteristics(indices, df_dataset, text_col, label_col):
    if df_dataset is None or not isinstance(indices, list) or not indices :
        return np.nan, np.nan, np.nan
    try:
        # Filtrar índices que podem não ser inteiros ou que são strings vazias
        valid_indices_str = [idx for idx in indices if str(idx).strip().isdigit()]
        if not valid_indices_str: return np.nan, np.nan, np.nan
        
        valid_indices = [int(i) for i in valid_indices_str if int(i) < len(df_dataset)]

    except (ValueError, TypeError) as e: 
        # print(f"Erro ao converter índices para L0: {indices} - {e}")
        return np.nan, np.nan, np.nan
        
    if not valid_indices: return np.nan, np.nan, np.nan

    l0_subset = df_dataset.iloc[valid_indices]
    total_tokens = l0_subset[text_col].astype(str).apply(lambda x: len(x.split())).sum()
    all_text = " ".join(l0_subset[text_col].astype(str).tolist())
    distinct_tokens = len(set(all_text.lower().split())) 
    num_unique_classes = l0_subset[label_col].nunique()
    return total_tokens, distinct_tokens, num_unique_classes

def load_detailed_log_raw(l0_size, metric_short_name, goal_short_name, max_gens=None): # Adicionado max_gens
    filepath = get_ag_detailed_fitness_filepath(l0_size, metric_short_name, goal_short_name)
    metric_col_in_file = 'accuracy_on_full' if metric_short_name.upper() == 'ACCURACY' else 'f1_macro_on_full'
    char_cols_from_log = ['num_tokens', 'num_distinct_tokens', 'num_classes_in_l0']
    try:
        df_detailed = pd.read_csv(filepath)
        if max_gens and 'generation' in df_detailed.columns: # TRUNCAR GERAÇÕES
            df_detailed = df_detailed[df_detailed['generation'] <= max_gens]

        if df_detailed.empty: return None
        required_cols = [metric_col_in_file, 'generation'] + char_cols_from_log
        if not all(col in df_detailed.columns for col in required_cols): return None
        df_detailed[metric_col_in_file] = pd.to_numeric(df_detailed[metric_col_in_file], errors='coerce')
        for char_col in char_cols_from_log:
            df_detailed[char_col] = pd.to_numeric(df_detailed[char_col], errors='coerce')
        df_detailed.dropna(subset=[metric_col_in_file, 'generation'] + char_cols_from_log, inplace=True) 
        if df_detailed.empty: return None
        return df_detailed[['generation', metric_col_in_file] + char_cols_from_log]
    except FileNotFoundError: return None
    except Exception: return None

# -

# ## 5. Evolução da População do AG por Tamanho de L0
# (Usa a nova função `plot_population_evolution_combined`)

# # +
# colors_max_palette = sns.color_palette("Greens_d", 3) 
# colors_min_palette = sns.color_palette("Oranges_d", 3) 

# if not CONVERGENCE_L0_SIZES_EXAMPLE:
#     print("Nenhum L0_SIZE definido em CONVERGENCE_L0_SIZES_EXAMPLE. Pulando seção de Evolução da População.")
# else:
#     for l0_size_conv in CONVERGENCE_L0_SIZES_EXAMPLE:
#         print(f"\n--- Gerando gráficos de Evolução da População para L0 = {l0_size_conv} ---")

#         # --- GRÁFICO PARA ACURÁCIA ---
#         metric_s_acc = "ACCURACY"
#         metric_n_acc = METRIC_MAP[metric_s_acc]
        
#         df_agg_acc_max_data = load_and_aggregate_detailed_log(l0_size_conv, metric_s_acc, "MAXIMIZE")
#         df_agg_acc_min_data = load_and_aggregate_detailed_log(l0_size_conv, metric_s_acc, "MINIMIZE")

#         plot_population_evolution_combined(
#             l0_size=l0_size_conv,
#             metric_name_display=metric_n_acc,
#             df_agg_max=df_agg_acc_max_data,
#             df_agg_min=df_agg_acc_min_data,
#             colors_max=colors_max_palette,
#             colors_min=colors_min_palette
#         )

#         # --- GRÁFICO PARA F1-SCORE ---
#         metric_s_f1 = "F1"
#         metric_n_f1 = METRIC_MAP[metric_s_f1]

#         df_agg_f1_max_data = load_and_aggregate_detailed_log(l0_size_conv, metric_s_f1, "MAXIMIZE")
#         df_agg_f1_min_data = load_and_aggregate_detailed_log(l0_size_conv, metric_s_f1, "MINIMIZE")

#         plot_population_evolution_combined(
#             l0_size=l0_size_conv,
#             metric_name_display=metric_n_f1,
#             df_agg_max=df_agg_f1_max_data,
#             df_agg_min=df_agg_f1_min_data,
#             colors_max=colors_max_palette,
#             colors_min=colors_min_palette
#         )
# # -

# ## 5.B Variante: Evolução da População Lado a Lado (Acurácia vs F1-Score) com Áreas
# (SEÇÃO MODIFICADA CONFORME ORIENTAÇÕES)

# +
colors_max_evo = sns.color_palette("Greens_d", 3) 
colors_min_evo = sns.color_palette("Blues_d", 3)  
fill_alpha_evo = 0.2

if not CONVERGENCE_L0_SIZES_EXAMPLE:
    print("Nenhum L0_SIZE definido em CONVERGENCE_L0_SIZES_EXAMPLE. Pulando seção 5.B.")
else:
    for l0_size_sls in CONVERGENCE_L0_SIZES_EXAMPLE: 
        print(f"\n--- L0 = {l0_size_sls}: Evolução Lado a Lado (Acurácia vs F1-Score) até {MAX_GENERATIONS_TO_PLOT} Gerações ---")
        
        fig_sls, axes_sls = plt.subplots(1, 2, figsize=(18, 6)) # Reduzido altura um pouco
        # fig_sls.suptitle(f"Evolução da Performance para L0 = {l0_size_sls} (até {MAX_GENERATIONS_TO_PLOT} Ger.)", fontsize=16)
        fig_sls.suptitle(f"Evolução da Performance para L0 = {l0_size_sls}", fontsize=16)

        # --- Subplot Esquerdo: ACURÁCIA ---
        ax_acc = axes_sls[0]
        metric_s_acc_sls = "ACCURACY"
        metric_n_acc_sls = METRIC_MAP[metric_s_acc_sls]
        
        df_acc_max_sls = load_and_aggregate_detailed_log(l0_size_sls, metric_s_acc_sls, "MAXIMIZE", max_gens=MAX_GENERATIONS_TO_PLOT)
        df_acc_min_sls = load_and_aggregate_detailed_log(l0_size_sls, metric_s_acc_sls, "MINIMIZE", max_gens=MAX_GENERATIONS_TO_PLOT)

        # Plot Maximização Acurácia
        if df_acc_max_sls is not None and not df_acc_max_sls.empty:
            gens_am = df_acc_max_sls['generation']
            ax_acc.fill_between(gens_am, df_acc_max_sls['min_metric'], df_acc_max_sls['max_metric'], color=colors_max_evo[1], alpha=fill_alpha_evo, label=f'Max Acc (Faixa Pop.)')
            ax_acc.plot(gens_am, df_acc_max_sls['avg_metric'], color=colors_max_evo[0], linestyle='--', label=f'Max Acc (Média Pop.)')
            ax_acc.plot(gens_am, df_acc_max_sls['max_metric'], color=colors_max_evo[2], linewidth=2, marker='.', markersize=4, label=f'Max Acc (Melhor Ind.)')

        # Plot Minimização Acurácia
        if df_acc_min_sls is not None and not df_acc_min_sls.empty:
            gens_ain = df_acc_min_sls['generation']
            ax_acc.fill_between(gens_ain, df_acc_min_sls['min_metric'], df_acc_min_sls['max_metric'], color=colors_min_evo[1], alpha=fill_alpha_evo, label=f'Min Acc (Faixa Pop.)')
            ax_acc.plot(gens_ain, df_acc_min_sls['avg_metric'], color=colors_min_evo[0], linestyle='--', label=f'Min Acc (Média Pop.)')
            ax_acc.plot(gens_ain, df_acc_min_sls['min_metric'], color=colors_min_evo[2], linewidth=2, marker='.', markersize=4, label=f'Min Acc (Pior Ind.)')
        
        ax_acc.set_title(f"Evolução da {metric_n_acc_sls}")
        ax_acc.set_xlabel("Geração")
        ax_acc.set_ylabel(f"{metric_n_acc_sls} (%)")
        ax_acc.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))
        ax_acc.legend(loc='best', fontsize=8)
        ax_acc.grid(True, which="both", ls="--", alpha=0.7)
        ax_acc.set_xlim(0, MAX_GENERATIONS_TO_PLOT + 1) # Limitar eixo X

        # Ajustar limites Y para centralizar
        if df_acc_max_sls is not None or df_acc_min_sls is not None:
            all_vals_acc = []
            if df_acc_max_sls is not None: all_vals_acc.extend(df_acc_max_sls[['min_metric', 'avg_metric', 'max_metric']].stack().tolist())
            if df_acc_min_sls is not None: all_vals_acc.extend(df_acc_min_sls[['min_metric', 'avg_metric', 'min_metric']].stack().tolist()) # Note: Pior indivíduo é min_metric para MIN
            if all_vals_acc:
                data_min_acc, data_max_acc = min(all_vals_acc), max(all_vals_acc)
                padding_acc = (data_max_acc - data_min_acc) * 0.1 # 10% de padding
                ax_acc.set_ylim(max(0, data_min_acc - padding_acc), min(1, data_max_acc + padding_acc))


        # --- Subplot Direito: F1-SCORE ---
        ax_f1 = axes_sls[1]
        metric_s_f1_sls = "F1"
        metric_n_f1_sls = METRIC_MAP[metric_s_f1_sls]

        df_f1_max_sls = load_and_aggregate_detailed_log(l0_size_sls, metric_s_f1_sls, "MAXIMIZE", max_gens=MAX_GENERATIONS_TO_PLOT)
        df_f1_min_sls = load_and_aggregate_detailed_log(l0_size_sls, metric_s_f1_sls, "MINIMIZE", max_gens=MAX_GENERATIONS_TO_PLOT)

        if df_f1_max_sls is not None and not df_f1_max_sls.empty:
            gens_fm = df_f1_max_sls['generation']
            ax_f1.fill_between(gens_fm, df_f1_max_sls['min_metric'], df_f1_max_sls['max_metric'], color=colors_max_evo[1], alpha=fill_alpha_evo, label=f'Max F1 (Faixa Pop.)')
            ax_f1.plot(gens_fm, df_f1_max_sls['avg_metric'], color=colors_max_evo[0], linestyle='--', label=f'Max F1 (Média Pop.)')
            ax_f1.plot(gens_fm, df_f1_max_sls['max_metric'], color=colors_max_evo[2], linewidth=2, marker='.', markersize=4, label=f'Max F1 (Melhor Ind.)')

        if df_f1_min_sls is not None and not df_f1_min_sls.empty:
            gens_fin = df_f1_min_sls['generation']
            ax_f1.fill_between(gens_fin, df_f1_min_sls['min_metric'], df_f1_min_sls['max_metric'], color=colors_min_evo[1], alpha=fill_alpha_evo, label=f'Min F1 (Faixa Pop.)')
            ax_f1.plot(gens_fin, df_f1_min_sls['avg_metric'], color=colors_min_evo[0], linestyle='--', label=f'Min F1 (Média Pop.)')
            ax_f1.plot(gens_fin, df_f1_min_sls['min_metric'], color=colors_min_evo[2], linewidth=2, marker='.', markersize=4, label=f'Min F1 (Pior Ind.)')

        ax_f1.set_title(f"Evolução do {metric_n_f1_sls}")
        ax_f1.set_xlabel("Geração")
        ax_f1.set_ylabel(f"{metric_n_f1_sls} (%)")
        ax_f1.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))
        ax_f1.legend(loc='best', fontsize=8)
        ax_f1.grid(True, which="both", ls="--", alpha=0.7)
        ax_f1.set_xlim(0, MAX_GENERATIONS_TO_PLOT + 1)

        if df_f1_max_sls is not None or df_f1_min_sls is not None:
            all_vals_f1 = []
            if df_f1_max_sls is not None: all_vals_f1.extend(df_f1_max_sls[['min_metric', 'avg_metric', 'max_metric']].stack().tolist())
            if df_f1_min_sls is not None: all_vals_f1.extend(df_f1_min_sls[['min_metric', 'avg_metric', 'min_metric']].stack().tolist())
            if all_vals_f1:
                data_min_f1, data_max_f1 = min(all_vals_f1), max(all_vals_f1)
                padding_f1 = (data_max_f1 - data_min_f1) * 0.1
                ax_f1.set_ylim(max(0, data_min_f1 - padding_f1), min(1, data_max_f1 + padding_f1))
        
        plt.tight_layout(rect=[0, 0, 1, 0.94]) # Ajuste para suptitle
        plt.show()
# -

# # ## 6. Comparação da "Curva Ótima" do AG com Amostragem Aleatória
# # (EIXO X EM ESCALA LINEAR)

# # +
# ag_optimal_data = {'L0_Size': [], 'Metric': [], 'Value': []}
# if not L0_SIZES_FOR_PLOTS:
#     print("Nenhum L0_SIZE definido em L0_SIZES_FOR_PLOTS. Pulando seção 'Curva Ótima'.")
# else:
#     for l0_size_plot in L0_SIZES_FOR_PLOTS: 
#         best_l0_acc_path = get_ag_best_l0_filepath(l0_size_plot, "ACCURACY", "MAXIMIZE")
#         try:
#             df_best_acc = pd.read_csv(best_l0_acc_path)
#             if not df_best_acc.empty and 'metric_value' in df_best_acc.columns:
#                 ag_optimal_data['L0_Size'].append(l0_size_plot); ag_optimal_data['Metric'].append('Acurácia'); ag_optimal_data['Value'].append(df_best_acc['metric_value'].iloc[0]) 
#             else: ag_optimal_data['L0_Size'].append(l0_size_plot); ag_optimal_data['Metric'].append('Acurácia'); ag_optimal_data['Value'].append(np.nan)
#         except Exception: ag_optimal_data['L0_Size'].append(l0_size_plot); ag_optimal_data['Metric'].append('Acurácia'); ag_optimal_data['Value'].append(np.nan)

#         best_l0_f1_path = get_ag_best_l0_filepath(l0_size_plot, "F1", "MAXIMIZE")
#         try:
#             df_best_f1 = pd.read_csv(best_l0_f1_path)
#             if not df_best_f1.empty and 'metric_value' in df_best_f1.columns:
#                 ag_optimal_data['L0_Size'].append(l0_size_plot); ag_optimal_data['Metric'].append('Macro F1-Score'); ag_optimal_data['Value'].append(df_best_f1['metric_value'].iloc[0])
#             else: ag_optimal_data['L0_Size'].append(l0_size_plot); ag_optimal_data['Metric'].append('Macro F1-Score'); ag_optimal_data['Value'].append(np.nan)
#         except Exception: ag_optimal_data['L0_Size'].append(l0_size_plot); ag_optimal_data['Metric'].append('Macro F1-Score'); ag_optimal_data['Value'].append(np.nan)
            
#     df_ag_optimal = pd.DataFrame(ag_optimal_data)
#     df_ag_optimal.dropna(subset=['Value'], inplace=True) 

#     if df_random_stats is not None and not df_ag_optimal.empty:
#         fig, axes = plt.subplots(2, 1, figsize=(12, 10), sharex=True) # sharex mantido, pois L0_Size é o mesmo
#         fig.suptitle('Performance dos L0s Ótimos do AG vs. Amostragem Aleatória', fontsize=16)

#         # Acurácia
#         ax = axes[0]; metric_disp = 'Acurácia'; rm, rmin, rmax = 'mean_accuracy', 'min_accuracy', 'max_accuracy'
#         if all(c in df_random_stats.columns for c in [rm, rmin, rmax]):
#             # Filtrar df_random_stats para os L0_SIZES_FOR_PLOTS para consistência no eixo X
#             df_random_subset_acc = df_random_stats[df_random_stats.index.isin(L0_SIZES_FOR_PLOTS)]
#             ax.plot(df_random_subset_acc.index, df_random_subset_acc[rm], label='Amost. Aleatória (Média)', marker='o', linestyle='--')
#             ax.fill_between(df_random_subset_acc.index, df_random_subset_acc[rmin], df_random_subset_acc[rmax], alpha=0.2, label='Amost. Aleatória (Faixa Min-Max)')
        
#         ag_sub = df_ag_optimal[df_ag_optimal['Metric'] == metric_disp]
#         if not ag_sub.empty: ax.plot(ag_sub['L0_Size'], ag_sub['Value'], label='AG (L0 Ótimo)', marker='s', color='red')
#         ax.set_ylabel(metric_disp + " (%)"); ax.legend(loc='best'); ax.grid(True, alpha=0.7, ls="--")
#         # ax.set_xscale('linear') # MUDANÇA AQUI - Removido 'log'
#         ax.set_xticks(L0_SIZES_FOR_PLOTS) 
#         ax.xaxis.set_major_formatter(mticker.ScalarFormatter()) # Formato normal para ticks
#         ax.set_title('Comparativo de Acurácia')
#         ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))


#         # F1-Score
#         ax = axes[1]; metric_disp = 'Macro F1-Score'; rm, rmin, rmax = 'mean_f1_score', 'min_f1_score', 'max_f1_score'
#         if all(c in df_random_stats.columns for c in [rm, rmin, rmax]):
#             df_random_subset_f1 = df_random_stats[df_random_stats.index.isin(L0_SIZES_FOR_PLOTS)]
#             ax.plot(df_random_subset_f1.index, df_random_subset_f1[rm], label='Amost. Aleatória (Média)', marker='o', linestyle='--')
#             ax.fill_between(df_random_subset_f1.index, df_random_subset_f1[rmin], df_random_subset_f1[rmax], alpha=0.2, label='Amost. Aleatória (Faixa Min-Max)')
        
#         ag_sub = df_ag_optimal[df_ag_optimal['Metric'] == metric_disp]
#         if not ag_sub.empty: ax.plot(ag_sub['L0_Size'], ag_sub['Value'], label='AG (L0 Ótimo)', marker='s', color='green')
#         ax.set_ylabel(metric_disp + " (%)"); ax.legend(loc='best'); ax.grid(True, alpha=0.7, ls="--")
#         ax.set_xlabel("Tamanho de L0 (I)")
#         # ax.set_xscale('linear') # MUDANÇA AQUI - Removido 'log'
#         ax.set_xticks(L0_SIZES_FOR_PLOTS) 
#         ax.xaxis.set_major_formatter(mticker.ScalarFormatter())
#         ax.set_title('Comparativo de Macro F1-Score')
#         ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))
        
#         plt.text(0.99, -0.15, "Ref: fig:ag_curva_otima_vs_aleatoria", ha='right', va='bottom', transform=axes[1].transAxes, fontsize=9, color='gray')
#         plt.tight_layout(rect=[0, 0.03, 1, 0.95]); plt.show()
#     elif df_random_stats is None: print("Dados de amostragem aleatória (df_random_stats) não disponíveis para 'Curva Ótima'.")
#     elif df_ag_optimal.empty : print("Dados otimizados do AG (df_ag_optimal) não disponíveis para 'Curva Ótima'.")

# # -

# # ## 7. Análise de Características dos L0s com Desempenho Extremo
# # (Mantido como antes, mas verificando `L0_SIZES_FOR_PLOTS`)

# # +
# characteristics_data_list = []

# if df_full is not None and L0_SIZES_FOR_PLOTS:
#     for l0_size_plot_char in L0_SIZES_FOR_PLOTS:
#         if df_random_stats is not None and l0_size_plot_char in df_random_stats.index:
#             rand_tt = df_random_stats.loc[l0_size_plot_char, 'avg_total_tokens'] if 'avg_total_tokens' in df_random_stats.columns else np.nan
#             rand_dt = df_random_stats.loc[l0_size_plot_char, 'avg_distinct_tokens'] if 'avg_distinct_tokens' in df_random_stats.columns else np.nan
#             rand_nc = df_random_stats.loc[l0_size_plot_char, 'avg_num_classes'] if 'avg_num_classes' in df_random_stats.columns else np.nan
#             characteristics_data_list.append({'L0_Size': l0_size_plot_char, 'Type': 'Amostragem Aleatória (Média)', 'Total Tokens': rand_tt, 'Tokens Distintos': rand_dt, 'Classes Únicas': rand_nc})
        
#         ag_l0_configs = [
#             {'metric_short': 'ACCURACY', 'goal_short': 'MAXIMIZE', 'label': 'AG Max Acurácia (Ótimo)'}, {'metric_short': 'ACCURACY', 'goal_short': 'MINIMIZE', 'label': 'AG Min Acurácia (Subótimo)'},
#             {'metric_short': 'F1', 'goal_short': 'MAXIMIZE', 'label': 'AG Max F1-Score (Ótimo)'}, {'metric_short': 'F1', 'goal_short': 'MINIMIZE', 'label': 'AG Min F1-Score (Subótimo)'},
#         ]
#         for config in ag_l0_configs:
#             l0_indices = []; filepath = get_ag_best_l0_filepath(l0_size_plot_char, config['metric_short'], config['goal_short'])
#             try:
#                 df_best = pd.read_csv(filepath)
#                 if not df_best.empty and 'l0_indices_str' in df_best.columns:
#                     indices_str = df_best['l0_indices_str'].iloc[0]
#                     if pd.notna(indices_str) and isinstance(indices_str, str): l0_indices = indices_str.split(',') # Mantem como string para get_l0_characteristics
#             except Exception: pass 
#             tt, dt, nc = get_l0_characteristics(l0_indices, df_full, TEXT_COLUMN, LABEL_COLUMN)
#             characteristics_data_list.append({'L0_Size': l0_size_plot_char, 'Type': config['label'], 'Total Tokens': tt, 'Tokens Distintos': dt, 'Classes Únicas': nc})

#     df_characteristics = pd.DataFrame(characteristics_data_list)
#     df_characteristics.dropna(subset=['Total Tokens', 'Tokens Distintos', 'Classes Únicas'], how='all', inplace=True)

#     if not df_characteristics.empty:
#         char_plot_configs = [
#             {'char_col': 'Tokens Distintos', 'title_suffix': 'Tamanho do Vocabulário', 'ylabel': 'Nº de Tokens Distintos', 'fig_label': 'ag_caracteristica_vocab_size'},
#             {'char_col': 'Total Tokens', 'title_suffix': 'Número Total de Tokens', 'ylabel': 'Nº Total de Tokens', 'fig_label': 'ag_caracteristica_total_tokens'},
#             {'char_col': 'Classes Únicas', 'title_suffix': 'Número de Classes Únicas', 'ylabel': 'Nº de Classes Únicas', 'fig_label': 'ag_caracteristica_num_classes'},
#         ]
#         for config in char_plot_configs:
#             plt.figure(figsize=(14, 8))
#             valid_types = df_characteristics.dropna(subset=[config['char_col']])['Type'].unique()
#             plot_df = df_characteristics[df_characteristics['Type'].isin(valid_types)]
#             if not plot_df.empty:
#                 all_types_ordered = ['AG Max Acurácia (Ótimo)', 'AG Max F1-Score (Ótimo)', 'AG Min Acurácia (Subótimo)', 'AG Min F1-Score (Subótimo)', 'Amostragem Aleatória (Média)']
#                 hue_order = [t for t in all_types_ordered if t in valid_types]
#                 sns.lineplot(data=plot_df, x='L0_Size', y=config['char_col'], hue='Type', marker='o', style='Type', dashes=True, hue_order=hue_order)
#                 plt.title(f"{config['title_suffix']} em L0s Ótimos, Subótimos e Aleatórios", fontsize=16)
#                 plt.xlabel("Tamanho de L0 (I)"); plt.ylabel(config['ylabel']); plt.xscale('log')
#                 plt.xticks(L0_SIZES_FOR_PLOTS); plt.gca().xaxis.set_major_formatter(mticker.ScalarFormatter())
#                 plt.legend(title='Tipo de L0', loc='center left', bbox_to_anchor=(1, 0.5))
#                 plt.grid(True, alpha=0.7, ls="--"); plt.text(0.99, 0.01, f"Ref: {config['fig_label']}", ha='right', va='bottom', transform=plt.gca().transAxes, fontsize=9, color='gray')
#                 plt.tight_layout(rect=[0, 0, 0.78, 1]); plt.show()
#             # else: print(f"Nenhum dado válido para plotar característica: {config['char_col']}")
#     # else: print("DataFrame de características vazio ou sem dados válidos após filtragem.")
# elif df_full is None: print("\nAVISO: Dataset completo (df_full) não carregado. Análise de características não executada.")
# elif not L0_SIZES_FOR_PLOTS: print("\nAVISO: Nenhum L0_SIZE em L0_SIZES_FOR_PLOTS. Análise de características não executada.")

# # -

# # ## 8. Gráficos de Evolução: Performance vs. Características do Melhor/Pior Indivíduo
# if not CONVERGENCE_L0_SIZES_EXAMPLE:
#     print("Nenhum L0_SIZE definido em CONVERGENCE_L0_SIZES_EXAMPLE. Pulando seção 8.B.")
# else:
#     char_cols_to_plot_y_axis = {
#         'num_tokens': 'Nº Total de Tokens no L0',
#         'num_distinct_tokens': 'Nº Tokens Distintos no L0',
#         'num_classes_in_l0': 'Nº Classes Únicas no L0'
#     }
#     main_metrics_to_plot = [
#         ("ACCURACY", METRIC_MAP["ACCURACY"]),
#         ("F1", METRIC_MAP["F1"])
#     ]

#     for l0_size_evol in CONVERGENCE_L0_SIZES_EXAMPLE:
#         print(f"\n--- L0 = {l0_size_evol}: Evolução Características (Y) vs. Geração (X), Cor: Performance ---")
        
#         for metric_short_b, metric_display_b in main_metrics_to_plot:
#             perf_col_for_color_b = 'accuracy_on_full' if metric_short_b == 'ACCURACY' else 'f1_macro_on_full'

#             for char_col_y_b, char_label_y_b in char_cols_to_plot_y_axis.items():
#                 df_min_raw = load_detailed_log_raw(l0_size_evol, metric_short_b, "MINIMIZE")
#                 df_max_raw = load_detailed_log_raw(l0_size_evol, metric_short_b, "MAXIMIZE")

#                 min_has_data = df_min_raw is not None and not df_min_raw.empty and char_col_y_b in df_min_raw.columns and not df_min_raw[char_col_y_b].isnull().all()
#                 max_has_data = df_max_raw is not None and not df_max_raw.empty and char_col_y_b in df_max_raw.columns and not df_max_raw[char_col_y_b].isnull().all()

#                 if not min_has_data and not max_has_data:
#                     continue

#                 fig, axes = plt.subplots(1, 2, figsize=(18, 7), sharey=True)
#                 fig.suptitle(f"L0={l0_size_evol}: Evolução de '{char_label_y_b.split(' no ')[0]}' (Cor: {metric_display_b})", fontsize=16)

#                 # Determinar vmin e vmax GLOBAIS para este PAR de subplots
#                 current_perf_values = []
#                 if min_has_data: current_perf_values.extend(df_min_raw[perf_col_for_color_b].dropna().tolist())
#                 if max_has_data: current_perf_values.extend(df_max_raw[perf_col_for_color_b].dropna().tolist())
                
#                 if not current_perf_values: # Caso raro, mas para evitar erro se ambos os DFs ficarem vazios após dropna
#                     vmin_global_pair = 0
#                     vmax_global_pair = 1
#                 else:
#                     vmin_global_pair = min(current_perf_values)
#                     vmax_global_pair = max(current_perf_values)
                
#                 # Se vmin e vmax forem muito próximos ou iguais, ajuste para evitar problemas com colorbar
#                 if np.isclose(vmin_global_pair, vmax_global_pair):
#                     vmax_global_pair = vmin_global_pair + 0.01 # Adiciona uma pequena margem
#                     if vmax_global_pair > 1.0 : vmax_global_pair = 1.0 # Limita a 1
#                     if vmin_global_pair < 0.0 : vmin_global_pair = 0.0 # Limita a 0


#                 norm_color_pair = plt.Normalize(vmin=vmin_global_pair, vmax=vmax_global_pair)
#                 cmap_choice = 'coolwarm' # Usaremos o mesmo cmap, a interpretação da cor depende do objetivo (min/max)

#                 # Subplot Esquerdo: Minimização
#                 ax_min = axes[0]
#                 scatter_min = None # Inicializar para referência da colorbar
#                 if min_has_data:
#                     sample_min_df = df_min_raw.sample(n=min(3000, len(df_min_raw)), random_state=42) if len(df_min_raw) > 3000 else df_min_raw
#                     scatter_min = ax_min.scatter(
#                         x=sample_min_df['generation'], y=sample_min_df[char_col_y_b],
#                         c=sample_min_df[perf_col_for_color_b], cmap=cmap_choice, 
#                         norm=norm_color_pair, alpha=0.6, s=30)
#                     ax_min.set_title(f"Minimização de {metric_display_b}")
#                     ax_min.set_xlabel("Geração")
#                     ax_min.set_ylabel(char_label_y_b)
#                     ax_min.grid(True, linestyle='--', alpha=0.6)
#                 else:
#                     ax_min.text(0.5, 0.5, "Sem dados", ha='center', va='center', transform=ax_min.transAxes)
#                     ax_min.set_title(f"Minimização de {metric_display_b}")

#                 # Subplot Direito: Maximização
#                 ax_max = axes[1]
#                 scatter_max = None
#                 if max_has_data:
#                     sample_max_df = df_max_raw.sample(n=min(3000, len(df_max_raw)), random_state=42) if len(df_max_raw) > 3000 else df_max_raw
#                     scatter_max = ax_max.scatter(
#                         x=sample_max_df['generation'], y=sample_max_df[char_col_y_b],
#                         c=sample_max_df[perf_col_for_color_b], cmap=cmap_choice, 
#                         norm=norm_color_pair, alpha=0.6, s=30)
#                     ax_max.set_title(f"Maximização de {metric_display_b}")
#                     ax_max.set_xlabel("Geração")
#                     ax_max.grid(True, linestyle='--', alpha=0.6)
#                 else:
#                     ax_max.text(0.5, 0.5, "Sem dados", ha='center', va='center', transform=ax_max.transAxes)
#                     ax_max.set_title(f"Maximização de {metric_display_b}")
                
#                 # Adicionar Colorbar ÚNICA à direita do subplot da direita
#                 # Usar o scatter_max se existir, senão o scatter_min (se este existir) para a colorbar.
#                 # Se ambos tiverem dados, a norm_color_pair garante que a escala é a mesma.
#                 mappable_for_colorbar = scatter_max if max_has_data else scatter_min
#                 if mappable_for_colorbar:
#                     cbar = fig.colorbar(mappable_for_colorbar, ax=axes, # Passar ambos os eixos para posicionamento correto
#                                         label=f"{metric_display_b} (%)", 
#                                         format=mticker.PercentFormatter(xmax=1.0, decimals=0),
#                                         fraction=0.03, pad=0.02) # Ajustar fraction e pad
                
#                 plt.tight_layout(rect=[0, 0, 1, 0.95]) 
#                 plt.show()

# # ## 8.C Gráficos Consolidados: Evolução Característica (Y) vs. Geração (X), Cor: Performance - TODOS L0s
# # (NOVA SEÇÃO)

# # +
# if not CONVERGENCE_L0_SIZES_EXAMPLE:
#     print("Nenhum L0_SIZE definido em CONVERGENCE_L0_SIZES_EXAMPLE. Pulando seção 8.C.")
# else:
#     # Características a plotar no eixo Y
#     char_cols_consolidated_y = {
#         'num_tokens': 'Nº Total de Tokens no L0',
#         'num_distinct_tokens': 'Nº Tokens Distintos no L0',
#         'num_classes_in_l0': 'Nº Classes Únicas no L0'
#     }
    
#     # Métricas principais (Acurácia, F1)
#     main_metrics_consolidated = [
#         ("ACCURACY", METRIC_MAP["ACCURACY"]),
#         ("F1", METRIC_MAP["F1"])
#     ]

#     for metric_short_c, metric_display_c in main_metrics_consolidated:
#         perf_col_color_c = 'accuracy_on_full' if metric_short_c == 'ACCURACY' else 'f1_macro_on_full'

#         for char_col_y_c, char_label_y_c in char_cols_consolidated_y.items():
            
#             fig_c, axes_c = plt.subplots(1, 2, figsize=(20, 8), sharey=True) # Mais largo para acomodar legenda de L0s
#             fig_c.suptitle(f"Consolidado: Evolução de '{char_label_y_c.split(' no ')[0]}' (Cor: {metric_display_c})", fontsize=18)

#             # Listas para coletar todos os dados para normalização global da colorbar (opcional, mas bom para comparação)
#             all_perf_values_min = []
#             all_perf_values_max = []

#             # Coletar dados primeiro para determinar range da colorbar global
#             for i, l0_size_c in enumerate(CONVERGENCE_L0_SIZES_EXAMPLE):
#                 df_min_raw_c = load_detailed_log_raw(l0_size_c, metric_short_c, "MINIMIZE")
#                 df_max_raw_c = load_detailed_log_raw(l0_size_c, metric_short_c, "MAXIMIZE")
#                 if df_min_raw_c is not None and not df_min_raw_c.empty and perf_col_color_c in df_min_raw_c.columns:
#                     all_perf_values_min.extend(df_min_raw_c[perf_col_color_c].dropna().tolist())
#                 if df_max_raw_c is not None and not df_max_raw_c.empty and perf_col_color_c in df_max_raw_c.columns:
#                     all_perf_values_max.extend(df_max_raw_c[perf_col_color_c].dropna().tolist())
            
#             global_min_perf = min(all_perf_values_min + all_perf_values_max) if (all_perf_values_min + all_perf_values_max) else 0
#             global_max_perf = max(all_perf_values_min + all_perf_values_max) if (all_perf_values_min + all_perf_values_max) else 1
#             norm_color_global = plt.Normalize(global_min_perf, global_max_perf)


#             # Subplot Esquerdo: Minimização Consolidado
#             ax_min_c = axes_c[0]
#             ax_min_c.set_title(f"Minimização de {metric_display_c} (Todos L0s)")
#             ax_min_c.set_xlabel("Geração")
#             ax_min_c.set_ylabel(char_label_y_c)
#             ax_min_c.grid(True, linestyle='--', alpha=0.6)
            
#             # Subplot Direito: Maximização Consolidado
#             ax_max_c = axes_c[1]
#             ax_max_c.set_title(f"Maximização de {metric_display_c} (Todos L0s)")
#             ax_max_c.set_xlabel("Geração")
#             ax_max_c.grid(True, linestyle='--', alpha=0.6)

#             legend_handles_min = []
#             legend_handles_max = []

#             for i, l0_size_c in enumerate(CONVERGENCE_L0_SIZES_EXAMPLE):
#                 marker_style = L0_MARKERS[i % len(L0_MARKERS)]
                
#                 # Minimização
#                 df_min_raw_c = load_detailed_log_raw(l0_size_c, metric_short_c, "MINIMIZE")
#                 if df_min_raw_c is not None and not df_min_raw_c.empty and char_col_y_c in df_min_raw_c.columns and not df_min_raw_c[char_col_y_c].isnull().all():
#                     sample_min_c_df = df_min_raw_c.sample(n=min(1000, len(df_min_raw_c)), random_state=42) if len(df_min_raw_c) > 1000 else df_min_raw_c # Amostragem menor por L0
#                     sc_min = ax_min_c.scatter(
#                         x=sample_min_c_df['generation'], y=sample_min_c_df[char_col_y_c],
#                         c=sample_min_c_df[perf_col_color_c], cmap='coolwarm_r', marker=marker_style,
#                         norm=norm_color_global, alpha=0.7, s=40, label=f"L0={l0_size_c}")
#                     if i == 0 : # Adicionar colorbar apenas uma vez por subplot
#                          cbar_min_c = fig_c.colorbar(sc_min, ax=ax_min_c, label=f"{metric_display_c} (%)", format=mticker.PercentFormatter(xmax=1.0, decimals=0), orientation='vertical', fraction=0.046, pad=0.04)


#                 # Maximização
#                 df_max_raw_c = load_detailed_log_raw(l0_size_c, metric_short_c, "MAXIMIZE")
#                 if df_max_raw_c is not None and not df_max_raw_c.empty and char_col_y_c in df_max_raw_c.columns and not df_max_raw_c[char_col_y_c].isnull().all():
#                     sample_max_c_df = df_max_raw_c.sample(n=min(1000, len(df_max_raw_c)), random_state=42) if len(df_max_raw_c) > 1000 else df_max_raw_c
#                     sc_max = ax_max_c.scatter(
#                         x=sample_max_c_df['generation'], y=sample_max_c_df[char_col_y_c],
#                         c=sample_max_c_df[perf_col_color_c], cmap='coolwarm', marker=marker_style,
#                         norm=norm_color_global, alpha=0.7, s=40, label=f"L0={l0_size_c}")
#                     if i == 0 :
#                         cbar_max_c = fig_c.colorbar(sc_max, ax=ax_max_c, label=f"{metric_display_c} (%)", format=mticker.PercentFormatter(xmax=1.0, decimals=0), orientation='vertical', fraction=0.046, pad=0.04)
            
#             # Adicionar legenda para os marcadores L0
#             # Devido à forma como scatter cria handles, é mais complexo criar uma legenda de marcadores.
#             # Uma abordagem é criar plots "invisíveis" ou usar PatchCollection.
#             # Por simplicidade, vamos criar manualmente os handles para a legenda.
#             handles = [plt.Line2D([0], [0], marker=L0_MARKERS[i % len(L0_MARKERS)], color='w', markerfacecolor='grey', markersize=7, label=f'L0={l0_s}') for i, l0_s in enumerate(CONVERGENCE_L0_SIZES_EXAMPLE)]
#             fig_c.legend(handles=handles, title="Tamanho de L0", loc='upper center', bbox_to_anchor=(0.5, 0.94), ncol=len(CONVERGENCE_L0_SIZES_EXAMPLE))
            
#             plt.tight_layout(rect=[0, 0, 1, 0.90]) # Ajuste para suptitle e legenda externa
#             plt.show()

# # ## 8. Gráficos de Evolução de Características (Layout 2x2)
# # (SEÇÃO MODIFICADA - Anteriormente 8.B)

# # +
# if not CONVERGENCE_L0_SIZES_EXAMPLE:
#     print("Nenhum L0_SIZE definido em CONVERGENCE_L0_SIZES_EXAMPLE. Pulando seção 8.")
# else:
#     char_cols_y_axis_8 = { # Características no Eixo Y
#         'num_tokens': 'Nº Total de Tokens no L0',
#         'num_distinct_tokens': 'Nº Tokens Distintos no L0',
#         'num_classes_in_l0': 'Nº Classes Únicas no L0'
#     }

#     for l0_size_8 in CONVERGENCE_L0_SIZES_EXAMPLE:
#         print(f"\n--- L0 = {l0_size_8}: Evolução Características vs. Geração (Cor: Performance) - Layout 2x2 ---")
        
#         for char_col_y_8, char_label_y_8 in char_cols_y_axis_8.items():
#             fig8, axes8 = plt.subplots(2, 2, figsize=(18, 12), sharex=True) # sharex para Geração
#             fig8.suptitle(f"L0={l0_size_8}: Evolução de '{char_label_y_8.split(' no ')[0]}' (até {MAX_GENERATIONS_TO_PLOT} Ger.)", fontsize=18)
            
#             plot_coords = [(0,0), (0,1), (1,0), (1,1)]
#             scenarios_8 = [
#                 ("ACCURACY", "MINIMIZE", "Minimização Acurácia"),
#                 ("ACCURACY", "MAXIMIZE", "Maximização Acurácia"),
#                 ("F1", "MINIMIZE", "Minimização F1-Score"),
#                 ("F1", "MAXIMIZE", "Maximização F1-Score")
#             ]

#             # Determinar vmin/vmax para Acurácia e F1 separadamente para este L0
#             perf_ranges = {"ACCURACY": [], "F1": []}
#             for metric_s_8, goal_s_8, _ in scenarios_8:
#                 df_raw_temp = load_detailed_log_raw(l0_size_8, metric_s_8, goal_s_8, max_gens=MAX_GENERATIONS_TO_PLOT)
#                 if df_raw_temp is not None and not df_raw_temp.empty:
#                     perf_col_temp = 'accuracy_on_full' if metric_s_8 == "ACCURACY" else 'f1_macro_on_full'
#                     if perf_col_temp in df_raw_temp.columns:
#                          perf_ranges[metric_s_8].extend(df_raw_temp[perf_col_temp].dropna().tolist())
            
#             norm_acc = None
#             if perf_ranges["ACCURACY"]:
#                 vmin_acc = min(perf_ranges["ACCURACY"])
#                 vmax_acc = max(perf_ranges["ACCURACY"])
#                 if np.isclose(vmin_acc, vmax_acc): vmax_acc = vmin_acc + 0.01
#                 norm_acc = plt.Normalize(vmin=vmin_acc, vmax=vmax_acc)

#             norm_f1 = None
#             if perf_ranges["F1"]:
#                 vmin_f1 = min(perf_ranges["F1"])
#                 vmax_f1 = max(perf_ranges["F1"])
#                 if np.isclose(vmin_f1, vmax_f1): vmax_f1 = vmin_f1 + 0.01
#                 norm_f1 = plt.Normalize(vmin=vmin_f1, vmax=vmax_f1)

#             has_any_subplot_data = False
#             for i, (metric_s_8, goal_s_8, scenario_label_8) in enumerate(scenarios_8):
#                 ax8 = axes8[plot_coords[i]]
#                 df_raw_8 = load_detailed_log_raw(l0_size_8, metric_s_8, goal_s_8, max_gens=MAX_GENERATIONS_TO_PLOT)

#                 if df_raw_8 is not None and not df_raw_8.empty and char_col_y_8 in df_raw_8.columns and not df_raw_8[char_col_y_8].isnull().all():
#                     has_any_subplot_data = True
#                     perf_col_color_8 = 'accuracy_on_full' if metric_s_8 == "ACCURACY" else 'f1_macro_on_full'
#                     perf_display_8 = METRIC_MAP[metric_s_8]
                    
#                     current_norm = norm_acc if metric_s_8 == "ACCURACY" else norm_f1
#                     current_cmap = 'coolwarm_r' if goal_s_8 == "MINIMIZE" else 'coolwarm' # Inverter para minimização

#                     sample_df_8 = df_raw_8.sample(n=min(1500, len(df_raw_8)), random_state=42) if len(df_raw_8) > 1500 else df_raw_8
                    
#                     if current_norm is None: # Se não houve dados para esta métrica em nenhum cenário
#                         print(f"    L0={l0_size_8}, {scenario_label_8}, Caract={char_label_y_8.split(' no ')[0]}: Sem dados para normalização da cor.")
#                         ax8.text(0.5, 0.5, "Sem dados de performance\npara a cor", ha='center', va='center', transform=ax8.transAxes, fontsize=9)

#                     sc = ax8.scatter(
#                         x=sample_df_8['generation'], y=sample_df_8[char_col_y_8],
#                         c=sample_df_8[perf_col_color_8] if current_norm else 'grey', # Cor cinza se não houver norma
#                         cmap=current_cmap if current_norm else None, 
#                         norm=current_norm, alpha=0.5, s=20)
                    
#                     ax8.set_title(scenario_label_8, fontsize=11)
#                     ax8.set_xlabel("Geração" if plot_coords[i][0] == 1 else "") # Label X só na linha de baixo
#                     ax8.set_ylabel(char_label_y_8 if plot_coords[i][1] == 0 else "") # Label Y só na coluna da esquerda
#                     ax8.grid(True, linestyle='--', alpha=0.6)
#                     ax8.set_xlim(0, MAX_GENERATIONS_TO_PLOT + 1)

#                     # Adicionar colorbar se este é o último subplot da linha com dados para esta métrica
#                     if plot_coords[i][1] == 1 and current_norm: # Última coluna
#                          fig8.colorbar(sc, ax=ax8, label=f"{perf_display_8} (%)", format=mticker.PercentFormatter(xmax=1.0, decimals=0), fraction=0.046, pad=0.06)
#                 else:
#                     ax8.text(0.5, 0.5, "Sem dados", ha='center', va='center', transform=ax8.transAxes, fontsize=10)
#                     ax8.set_title(scenario_label_8, fontsize=11)
#                     ax8.set_xlabel("Geração" if plot_coords[i][0] == 1 else "")
#                     ax8.set_ylabel(char_label_y_8 if plot_coords[i][1] == 0 else "")
            
#             if has_any_subplot_data:
#                 plt.tight_layout(rect=[0, 0, 1, 0.95])
#                 plt.show()
#             else:
#                 plt.close(fig8) # Fecha a figura se nenhum subplot teve dados
#                 print(f"    L0={l0_size_8}, Caract={char_label_y_8.split(' no ')[0]}: Nenhum dado para nenhum cenário.")

# ## 9. Resumo dos Resultados da Otimização
# (TABELA FINAL MODIFICADA)

# +
summary_table_data = []

if not AVAILABLE_L0_SIZES:
    print("Nenhum L0_SIZE encontrado em AVAILABLE_L0_SIZES. Pulando seção de Resumo Final.")
else:
    for l0_s_sum in AVAILABLE_L0_SIZES:
        row_data = {"L0 Size": l0_s_sum}
        
        # Acurácia
        df_acc_min_sum = load_and_aggregate_detailed_log(l0_s_sum, "ACCURACY", "MINIMIZE", max_gens=MAX_GENERATIONS_TO_PLOT)
        df_acc_max_sum = load_and_aggregate_detailed_log(l0_s_sum, "ACCURACY", "MAXIMIZE", max_gens=MAX_GENERATIONS_TO_PLOT)
        
        acc_min_val = df_acc_min_sum['min_metric'].iloc[-1] if df_acc_min_sum is not None and not df_acc_min_sum.empty else np.nan
        acc_max_val = df_acc_max_sum['max_metric'].iloc[-1] if df_acc_max_sum is not None and not df_acc_max_sum.empty else np.nan
        row_data["Acurácia (Min)"] = f"{acc_min_val*100:.2f}%" if pd.notna(acc_min_val) else "N/A"
        row_data["Acurácia (Max)"] = f"{acc_max_val*100:.2f}%" if pd.notna(acc_max_val) else "N/A"
        
        # F1-Score
        df_f1_min_sum = load_and_aggregate_detailed_log(l0_s_sum, "F1", "MINIMIZE", max_gens=MAX_GENERATIONS_TO_PLOT)
        df_f1_max_sum = load_and_aggregate_detailed_log(l0_s_sum, "F1", "MAXIMIZE", max_gens=MAX_GENERATIONS_TO_PLOT)

        f1_min_val = df_f1_min_sum['min_metric'].iloc[-1] if df_f1_min_sum is not None and not df_f1_min_sum.empty else np.nan
        f1_max_val = df_f1_max_sum['max_metric'].iloc[-1] if df_f1_max_sum is not None and not df_f1_max_sum.empty else np.nan
        row_data["F1-Score (Min)"] = f"{f1_min_val*100:.2f}%" if pd.notna(f1_min_val) else "N/A"
        row_data["F1-Score (Max)"] = f"{f1_max_val*100:.2f}%" if pd.notna(f1_max_val) else "N/A"

        # Gerações (usar a maior entre as execuções para este L0, até MAX_GENERATIONS_TO_PLOT)
        gens_acc_min = df_acc_min_sum['generation'].max() if df_acc_min_sum is not None and not df_acc_min_sum.empty else 0
        gens_acc_max = df_acc_max_sum['generation'].max() if df_acc_max_sum is not None and not df_acc_max_sum.empty else 0
        gens_f1_min  = df_f1_min_sum['generation'].max() if df_f1_min_sum is not None and not df_f1_min_sum.empty else 0
        gens_f1_max  = df_f1_max_sum['generation'].max() if df_f1_max_sum is not None and not df_f1_max_sum.empty else 0
        max_gen_l0 = max(gens_acc_min, gens_acc_max, gens_f1_min, gens_f1_max)
        row_data["Nº Gerações Exibidas"] = max_gen_l0 if max_gen_l0 > 0 else "N/A"
        
        summary_table_data.append(row_data)

    if summary_table_data:
        df_summary_final = pd.DataFrame(summary_table_data)
        print("\n--- Tabela Resumo Final da Performance (na última geração exibida) ---")
        try:
            from IPython.display import display, HTML
            display(HTML(df_summary_final.to_html(index=False)))
        except ImportError: 
            print(df_summary_final.to_string(index=False))
    else:
        print("Nenhum dado para gerar a tabela resumo final.")

# -

# ## 10. Fim da Análise